In [3]:
"""Finds best policy in the invManagement problem to yield highest profit.

On every iteration, improve priority_v1 over the priority_vX methods from previous iterations.
Make changes to yield higher profit.
Try to make the code short.
"""
from scipy.optimize import minimize
import or_gym
import numpy as np
import funsearch
@funsearch.run
def evaluate(n) -> int:
  """Generates a random knapsack problem with a fixed seed, solves it, and returns the maximum value."""
  results = solve()
  max_value = sum(results)/len(results)
  return int(max_value)




def solve():
  env_name='InvManagement-v1'
  env_config = {}
  results = priority(env_name, env_config)
  return results

def priority(env_name, env_config):
    """Improved version of `priority_v0` by using better optimization and policy tuning."""
    def dfo_func(policy, env, *args):
        env.reset()  # Ensure env is fresh
        rewards = []
        done = False
        while not done:
            action = base_stock_policy(policy, env)
            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break
        
        rewards = np.array(rewards)
        prob = env.demand_dist.pmf(env.D, **env.dist_param)
        
        # Return negative of expected profit
        return -1 / env.num_periods * np.sum(prob * rewards)
    
    def optimize_inventory_policy(env_name, fun,
        init_policy=None, env_config={}, method='Powell'):
        
        env = or_gym.make(env_name, env_config=env_config)
        
        if init_policy is None:
            init_policy = np.random.rand(env.num_stages - 1) * env.max_order
        else:
            init_policy = np.clip(init_policy, 0, env.max_order)
            
        # Use a more robust optimizer
        out = minimize(fun=fun, x0=init_policy, args=env, 
            method=method)
        policy = out.x.copy()
        
        # Policy must be positive integer
        policy = np.round(np.maximum(policy, 0), 0).astype(int)
        
        return policy, out
    
    def base_stock_policy(policy, env):  
        assert len(policy) == len(env.init_inv), (
          'Policy should match number of nodes in network' + 
          '({}, {}).'.format(len(policy), len(env.init_inv)))
        # Get echelon inventory levels
        if env.period == 0:
          inv_ech = np.cumsum(env.I[env.period] +
            env.T[env.period])
        else:
          inv_ech = np.cumsum(env.I[env.period] +
            env.T[env.period] - env.B[env.period-1, :-1])
        # Get unconstrained actions
        unc_actions = policy - inv_ech
        unc_actions = np.where(unc_actions > 0, unc_actions, 0)
        # Ensure that actions can be fulfilled by checking 
        # constraints
        inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
        actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
        return actions
    
    policy, out = optimize_inventory_policy(env_name, dfo_func)
    env = or_gym.make(env_name, env_config=env_config)
    eps = 1000
    rewards = []
    for i in range(eps):
        env.reset()
        reward = 0
        while True:
            action = base_stock_policy(policy, env)
            s, r, done, _ = env.step(action)
            reward += r
            if done:
                rewards.append(reward)
                break
    return rewards

In [4]:
rewards_v1 = priority('InvManagement-v1', {})
print(f"Averaged reward over episodes: {np.mean(rewards_v1)}")

AttributeError: 'InvManagementLostSalesEnv' object has no attribute 'max_order'